<a href="https://colab.research.google.com/github/Protron47/Sentiment_Analysis/blob/main/Sentiment_Analysis_Prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Analysis Prototype

**Deep Learning Approach for Sentiment Analysis**

This notebook contains a two-phase prototype you can run on Google Colab or locally:

- **Phase 1:** IMDb dataset with CNN and LSTM models

- **Phase 2:** Hybrid CNN-LSTM with **GloVe** embeddings for improved performance


In [ ]:
# --- Setup: Imports and basic settings ---
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
print('TensorFlow version:', tf.__version__)
seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)

# Parameters you can tweak
VOCAB_SIZE = 10000      # keep vocab small for speed (changeable)
MAX_LEN = 200           # max words per review (pad/truncate)
EMBED_DIM = 100         # embedding size when using trainable embedding
BATCH_SIZE = 128
EPOCHS = 3              # small for quick runs; increase for better results


TensorFlow version: 2.19.0


## Phase 1 — Load IMDb Dataset
Using the built-in Keras IMDb dataset (binary sentiment).

In [ ]:
# Load IMDb dataset
from tensorflow.keras.datasets import imdb
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)
word_index = imdb.get_word_index()
print('Train samples:', len(x_train), 'Test samples:', len(x_test))

# Map integer sequences back to words helper (optional)
index_to_word = {index+3: word for word, index in word_index.items()}
index_to_word[0] = '<PAD>'
index_to_word[1] = '<START>'
index_to_word[2] = '<UNK>'
index_to_word[3] = '<UNUSED>'

# Quick view
print('Example (integers):', x_train[0][:20])
print('Label:', y_train[0])

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train samples: 25000 Test samples: 25000
Example (integers): [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25]
Label: 1


## Preprocessing: Padding & Tokenization (already tokenized as integers)

In [ ]:
# Pad sequences
from tensorflow.keras.preprocessing.sequence import pad_sequences
x_train = pad_sequences(x_train, maxlen=MAX_LEN, padding='post', truncating='post')
x_test = pad_sequences(x_test, maxlen=MAX_LEN, padding='post', truncating='post')
print('Shape train:', x_train.shape, 'Shape test:', x_test.shape)

Shape train: (25000, 200) Shape test: (25000, 200)


## Model 1 — CNN (Phase 1)

In [ ]:
def build_cnn(vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, max_len=MAX_LEN):
    inputs = keras.Input(shape=(max_len,), dtype='int32')
    x = layers.Embedding(vocab_size, embed_dim, input_length=max_len)(inputs)
    x = layers.Conv1D(filters=128, kernel_size=5, activation='relu')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Conv1D(filters=128, kernel_size=5, activation='relu')(x)
    x = layers.GlobalMaxPooling1D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs, name='cnn_model')
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

cnn_model = build_cnn()
cnn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "cnn_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 200, 100)       │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 196, 128)       │        64,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 98, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 94, 128)        │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,162,817 (4.44 MB)

 Trainable params: 1,162,817 (4.44 MB)

 Non-trainable params: 0 (0.00 B)

### Train CNN (quick)

In [ ]:
history_cnn = cnn_model.fit(x_train, y_train,epochs=EPOCHS, batch_size=BATCH_SIZE,
                        validation_split=0.15)

NameError: name 'cnn_model' is not defined

## Model 2 — LSTM (Phase 1)

In [ ]:
def build_lstm(vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, max_len=MAX_LEN):
    inputs = keras.Input(shape=(max_len,), dtype='int32')
    x = layers.Embedding(vocab_size, embed_dim, input_length=max_len)(inputs)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=False))(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs, name='lstm_model')
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

lstm_model = build_lstm()
lstm_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "lstm_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 200, 100)       │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 256)            │       234,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,267,521 (4.84 MB)

 Trainable params: 1,267,521 (4.84 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_lstm = lstm_model.fit(x_train, y_train,
                        epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.15)

Epoch 1/3
167/167 ━━━━━━━━━━━━━━━━━━━━ 251s 1s/step - accuracy: 0.6285 - loss: 0.6151 - val_accuracy: 0.8448 - val_loss: 0.3956
Epoch 2/3
167/167 ━━━━━━━━━━━━━━━━━━━━ 247s 1s/step - accuracy: 0.8688 - loss: 0.3288 - val_accuracy: 0.8661 - val_loss: 0.3394
Epoch 3/3
167/167 ━━━━━━━━━━━━━━━━━━━━ 260s 2s/step - accuracy: 0.9067 - loss: 0.2518 - val_accuracy: 0.8227 - val_loss: 0.4093


## Evaluation — Phase 1
Evaluate both models on the test set and display confusion matrices and metrics.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def evaluate_model(model, x_test, y_test):
    preds = (model.predict(x_test) > 0.5).astype('int32').flatten()
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    cm = confusion_matrix(y_test, preds)
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'confusion_matrix': cm}

print('Evaluating CNN...')
cnn_metrics = evaluate_model(cnn_model, x_test, y_test)
print(cnn_metrics)
print('\nEvaluating LSTM...')
lstm_metrics = evaluate_model(lstm_model, x_test, y_test)
print(lstm_metrics)

Evaluating CNN...
782/782 ━━━━━━━━━━━━━━━━━━━━ 25s 32ms/step
{'accuracy': 0.8418, 'precision': 0.8364968102701426, 'recall': 0.84968, 'f1': 0.8430368694685876, 'confusion_matrix': array([[10424,  2076],
       [ 1879, 10621]])}

Evaluating LSTM...
782/782 ━━━━━━━━━━━━━━━━━━━━ 135s 172ms/step
{'accuracy': 0.81176, 'precision': 0.7833769633507853, 'recall': 0.86184, 'f1': 0.8207374676215146, 'confusion_matrix': array([[ 9521,  2979],
       [ 1727, 10773]])}


### Quick Inference: test with custom text (uses IMDb word index)

In [ ]:
def decode_review(ints):
    return ' '.join([index_to_word.get(i, '?') for i in ints if i > 3])

def prepare_text_for_model(text, word_index_map, max_len=MAX_LEN):
    # Simple tokenizer using the imdb word index mapping
    tokens = []
    for word in keras.preprocessing.text.text_to_word_sequence(text):
        idx = word_index_map.get(word)
        if idx is None or idx >= VOCAB_SIZE:
            tokens.append(2)  # <UNK>
        else:
            tokens.append(idx+3)
    return pad_sequences([tokens], maxlen=max_len, padding='post', truncating='post')

sample = "My mom bought silver worth one lakh eighty thousand."
x_sample = prepare_text_for_model(sample, word_index)
print('Sample tokens:', x_sample[0][:20])
print('CNN prediction (1=positive):', cnn_model.predict(x_sample)[0][0])
print('LSTM prediction (1=positive):', lstm_model.predict(x_sample)[0][0])

NameError: name 'MAX_LEN' is not defined

## Phase 2 — Hybrid CNN-LSTM with GloVe embeddings
This phase shows how to load pre-trained GloVe and build a hybrid model.

In [ ]:
USE_GLOVE = False  # set to True if you want to download and use GloVe

GLOVE_DIR = '/content/glove.6B'
GLOVE_FILE = 'glove.6B.100d.txt'  # corresponds to EMBED_DIM=100
if USE_GLOVE:
    print('Downloading GloVe (this runs on Colab or internet-enabled env)...')
    try:
        os.makedirs(GLOVE_DIR, exist_ok=True)
        !wget -q -O /content/glove.zip http://nlp.stanford.edu/data/glove.6B.zip
        !unzip -q /content/glove.zip -d /content/
        print('GloVe downloaded.')
    except Exception as e:
        print('Could not download GloVe in this environment:', e)

def build_embedding_matrix(word_index_map, vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM):
    embeddings_index = {}
    glove_path = os.path.join(GLOVE_DIR, GLOVE_FILE)
    if not os.path.exists(glove_path):
        print('GloVe file not found locally. Skipping pre-trained embeddings.')
        return None
    with open(glove_path, 'r', encoding='utf8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs
    print('Loaded GloVe vectors:', len(embeddings_index))
    embedding_matrix = np.random.normal(size=(vocab_size, embed_dim)).astype('float32')
    for word, i in word_index_map.items():
        idx = i + 3
        if idx < vocab_size:
            vec = embeddings_index.get(word)
            if vec is not None:
                embedding_matrix[idx] = vec
    return embedding_matrix

# To create embedding_matrix, run: emb = build_embedding_matrix(word_index)

### Hybrid CNN-LSTM Model (uses embedding layer — can use pre-trained GloVe if available)

In [ ]:
def build_hybrid(vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, max_len=MAX_LEN, embedding_matrix=None, trainable=True):
    inputs = keras.Input(shape=(max_len,), dtype='int32')
    if embedding_matrix is not None:
        emb_layer = layers.Embedding(vocab_size, embed_dim, weights=[embedding_matrix], input_length=max_len, trainable=trainable)
    else:
        emb_layer = layers.Embedding(vocab_size, embed_dim, input_length=max_len, trainable=True)
    x = emb_layer(inputs)
    x = layers.Conv1D(filters=128, kernel_size=5, activation='relu')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Bidirectional(layers.LSTM(128))(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs, name='hybrid_cnn_lstm')
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# Build and summarize (embedding_matrix can be created via build_embedding_matrix if GloVe was downloaded)
hybrid_model = build_hybrid()
hybrid_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "hybrid_cnn_lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 200, 100)       │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 196, 128)       │        64,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 98, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 256)            │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,360,321 (5.19 MB)

 Trainable params: 1,360,321 (5.19 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train hybrid (optional heavy)
# history_hybrid = hybrid_model.fit(x_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.15)

## Comparison & Save Models
Save trained models and show a summary comparison table.

In [ ]:
cnn_model.save('cnn_model.h5')
lstm_model.save('lstm_model.h5')
hybrid_model.save('hybrid_model.h5')

# If you trained all three, you can evaluate and compare:
cnn_metrics = evaluate_model(cnn_model, x_test, y_test)
lstm_metrics = evaluate_model(lstm_model, x_test, y_test)
hybrid_metrics = evaluate_model(hybrid_model, x_test, y_test)
print('CNN:', cnn_metrics)
print('LSTM:', lstm_metrics)
print('Hybrid:', hybrid_metrics)

print('Notebook complete. Train models by running cells sequentially.')

782/782 ━━━━━━━━━━━━━━━━━━━━ 26s 33ms/step
782/782 ━━━━━━━━━━━━━━━━━━━━ 140s 179ms/step
782/782 ━━━━━━━━━━━━━━━━━━━━ 91s 116ms/step
CNN: {'accuracy': 0.8418, 'precision': 0.8364968102701426, 'recall': 0.84968, 'f1': 0.8430368694685876, 'confusion_matrix': array([[10424,  2076],
       [ 1879, 10621]])}
LSTM: {'accuracy': 0.81176, 'precision': 0.7833769633507853, 'recall': 0.86184, 'f1': 0.8207374676215146, 'confusion_matrix': array([[ 9521,  2979],
       [ 1727, 10773]])}
Hybrid: {'accuracy': 0.50828, 'precision': 0.508320604550205, 'recall': 0.50584, 'f1': 0.507077268535226, 'confusion_matrix': array([[6384, 6116],
       [6177, 6323]])}
Notebook complete. Train models by running cells sequentially.


### Notes
- Use Colab GPU for faster training.\n- Increase `EPOCHS` for better performance (3 is set for quick demo).\n- To use GloVe: set `USE_GLOVE = True` and run the cell to download embeddings (Colab only).